### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\Admin\Desktop\Sona\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Admin\Desktop\Sona\RAG\notebook


In [3]:
### Read all the PDF files in the "pdfs" directory
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory and return a list of documents."""
    all_documents = []
    pdf_dir = Path(pdf_directory) 

    #Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(list(pdf_dir.glob("*")))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata 
            for doc in documents:
                doc.metadata["source_files"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print (f"Loaded {len(documents)} documents from {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the "pdfs" directory
all_pdfs_documents = process_all_pdfs(r"C:\Users\Admin\Desktop\Sona\RAG\data\pdf")

[WindowsPath('C:/Users/Admin/Desktop/Sona/RAG/data/pdf/SonalMittal.pdf'), WindowsPath('C:/Users/Admin/Desktop/Sona/RAG/data/pdf/Sonal_medicalReport.pdf')]
Found 2 PDF files in C:\Users\Admin\Desktop\Sona\RAG\data\pdf
Processing SonalMittal.pdf
Loaded 1 documents from SonalMittal.pdf
Processing Sonal_medicalReport.pdf
Loaded 31 documents from Sonal_medicalReport.pdf
Total documents loaded: 32


In [4]:
all_pdfs_documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-12T07:05:46+00:00', 'author': 'Sonal Mittal', 'keywords': '', 'moddate': '2026-02-12T07:05:46+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': 'Sonal Mittal Resume', 'trapped': '/False', 'source': 'C:\\Users\\Admin\\Desktop\\Sona\\RAG\\data\\pdf\\SonalMittal.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_files': 'SonalMittal.pdf', 'file_type': 'pdf'}, page_content='Sonal Mittal\n+91 7068229535 | sonalmittal140103@gmail.com | github.com/mittalsonal | linkedin.com/in/mittalsonal\nSUMMAR Y\nFull Stack Software Engineer with hands-on experience building scalable web applications and integrating AI/ML-driven features.\nProficient in Next.js, Node.js, Django, and MongoDB, with practical exposure to LLMs, LangChain, and Generative AI sys-\ntems. Strong foundation in REST API devel

### Now Chunking will be done 

In [5]:
### Text splitting get into chunks 
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200): 
    '''Split the documents into smaller chunks for better processing.'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show an example of the split documents
    if split_docs: 
        print("\nExample of a split document chunk:")
        print(f"Metadata: {split_docs[0].metadata}")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print the first 200 characters of the content

    return split_docs

In [6]:
chunks = split_documents(all_pdfs_documents)
chunks 

Split 32 documents into 69 chunks.

Example of a split document chunk:
Metadata: {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-12T07:05:46+00:00', 'author': 'Sonal Mittal', 'keywords': '', 'moddate': '2026-02-12T07:05:46+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': 'Sonal Mittal Resume', 'trapped': '/False', 'source': 'C:\\Users\\Admin\\Desktop\\Sona\\RAG\\data\\pdf\\SonalMittal.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_files': 'SonalMittal.pdf', 'file_type': 'pdf'}
Content: Sonal Mittal
+91 7068229535 | sonalmittal140103@gmail.com | github.com/mittalsonal | linkedin.com/in/mittalsonal
SUMMAR Y
Full Stack Software Engineer with hands-on experience building scalable web ap...


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-12T07:05:46+00:00', 'author': 'Sonal Mittal', 'keywords': '', 'moddate': '2026-02-12T07:05:46+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': 'Sonal Mittal Resume', 'trapped': '/False', 'source': 'C:\\Users\\Admin\\Desktop\\Sona\\RAG\\data\\pdf\\SonalMittal.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_files': 'SonalMittal.pdf', 'file_type': 'pdf'}, page_content='Sonal Mittal\n+91 7068229535 | sonalmittal140103@gmail.com | github.com/mittalsonal | linkedin.com/in/mittalsonal\nSUMMAR Y\nFull Stack Software Engineer with hands-on experience building scalable web applications and integrating AI/ML-driven features.\nProficient in Next.js, Node.js, Django, and MongoDB, with practical exposure to LLMs, LangChain, and Generative AI sys-\ntems. Strong foundation in REST API devel

### Embedding and Vector Store DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    '''Handles document embedding generation and storage using SentenceTransformer and ChromaDB.'''
    def __init__(self, model_name: str =  "all-MiniLM-L6-v2"):

        """Initialize the embedding manager with a specified model and ChromaDB client.
        Args:
            model_name : Hugging Face model name for generating embeddings. Default is "all-MiniLM-L6-v2", which is a compact and efficient model suitable for many applications.
        """

        self.model_name = model_name
        self.model = None
        self._load_model() #protected function to load the model

    def _load_model(self):
        """Load the specified SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts using the loaded model.
        Args:
            texts : A list of strings for which to generate embeddings.
            
        Returns:
            A numpy array of shape (len(texts), embedding_dimension) containing the generated embeddings.
        """

        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() before generating embeddings.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True) #encode is the method provided by sentence-transformers to generate embeddings for a list of texts. It returns a numpy array of shape (len(texts), embedding_dimension) containing the generated embeddings.

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


    # def get_sentence_embedding_dimension(self) -> int:
    #     """Get the dimensionality of the embeddings generated by the model.
    #     Returns:
    #         The dimension of the sentence embeddings as an integer.
    #     """
    #     if not self.model:
    #         raise ValueError("Model not loaded. Call _load_model() before getting embedding dimension.")
        
    #     return self.model.get_sentence_embedding_dimension()



# Intialize the embedding manager and generate embeddings for the document chunks

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2190.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [9]:
class VectorStore :
    '''Manages the storage and retrieval of document embeddings using ChromaDB.'''

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store/"):
        '''Intialize the vector store with a specified collection name and persistence directory.
        
        Args:
        collection_name : The name of the ChromaDB collection to use for storing document embeddings. Default is "pdf_documents".
        persist_directory : The directory where the ChromaDB collection will be persisted. Default is "../data/vector_store/".
        
        '''

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() #protected function to initialize the ChromaDB client and collection